In [1]:
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

In [2]:
import json
import ctypes
import struct
import blosc2
import numpy as np
import numpy as np
from pathlib import Path
import sys
sys.path.append('/projects/insituperf/SZ3/tools/pysz/')
from pysz import SZ

In [3]:
f = open('/projects/insituperf/seer_o/test_app/mochi-yokan-config.json')
json_data = json.load(f)
json_data

{'sim-id': '07111',
 'libraries': {'yokan': '/vast/home/pascalgrosset/spack/opt/spack/linux-rhel8-haswell/gcc-9.4.0/mochi-yokan-0.4.2-hiu7yh7om6nmyc2ahuknpdsov5k64zcj/lib/libyokan-bedrock-module.so'},
 'providers': [{'name': 'yokan_provider',
   'provider_id': 124,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map'}}}],
 'data': [{'name': 'pressure_3', 'compressor': 'BLOSC'},
  {'name': 'temperature_3', 'compressor': 'SZ3', 'psnr': 50}],
 'databases': [{'address': '192.168.81.72:46185',
   'protocol': 'ofi+tcp',
   'provider_id': 124}]}

{'sim-id': '07111',
 'libraries': {'yokan': '/vast/home/pascalgrosset/spack/opt/spack/linux-rhel8-haswell/gcc-9.4.0/mochi-yokan-0.4.2-hiu7yh7om6nmyc2ahuknpdsov5k64zcj/lib/libyokan-bedrock-module.so'},
 'providers': [{'name': 'yokan_provider',
   'provider_id': 124,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map'}}}],
 'data': [{'name': 'pressure_3', 'compressor': 'BLOSC'},
  {'name': 'temperature_3', 'compressor': 'SZ3', 'psnr': 50}],
 'databases': [{'address': '192.168.81.72:46185',
   'protocol': 'ofi+tcp',
   'provider_id': 124}]}

In [4]:
server_addr1 = "ofi+tcp://192.168.81.72:46185"
provider_id = 124
protocol = 'ofi+tcp'

In [5]:
engine1 = Engine(protocol)
mid1 = engine1.get_internal_mid()
addr1 = engine1.lookup(server_addr1)
hg_addr1 = addr1.get_internal_hg_addr()
provider1 = Provider(mid=mid1, provider_id=provider_id, config='{"database":{"type":"map"}}')
client1 = Client(mid=mid1)
db1 = client1.make_database_handle(address=hg_addr1, provider_id=provider_id)

In [6]:
dbs = []

In [7]:
dbs.append(db1)

In [47]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = '_07111_'
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [9]:
def split_key(key, pos):
    parts = key.split('/')
    name = parts[pos]
    return name

In [10]:
def get_field_name(key):
    parts = keys[0].split('/')
    name = parts[len(parts)-2]
    return name

In [11]:
def list_fields(db, timestep):
    
    all_keys = list_all_keys(db)
    print("all_keys:", all_keys)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[len(parts)-2]
        x.append(name)
    return list(set(x))

In [12]:
def list_attributes(db, key):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[2]
        field = parts[3]
        if name == key:
            x.append(field)
    x = list(set(x))

    return x

In [13]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [14]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [15]:
def get_decompDataBLOSC(db, key, num_elems):
        x = []
        
        val = get_data(db, key)
        a_bytesobj2 = blosc2.decompress(val)
        
        bf = str(num_elems) + 'f'
        x = struct.unpack(bf, a_bytesobj2)

        return x

In [16]:
def get_decompDataSZ3(dbs[0], key, num_elems):
        data = get_data(db, key)
        np_array = np.frombuffer(data, dtype=np.uint8)
        
        lib_extention = {
            "darwin": "libSZ3c.dylib",
            "windows": "SZ3c.dll",
        }.get(sys.platform, "libSZ3c.so")

        sz = SZ("/projects/insituperf/SZ3/install/lib64/{}".format(lib_extention))
        
        data_dec = sz.decompress(np_array, (num_elems,1,1), np.float32)


        return data_dec

In [17]:
ts = '_2'

In [48]:
keys = list_fields(dbs[0], ts)
keys

all_keys: ['_07111_0/0/pressure_3/compressed_size', '_07111_0/0/pressure_3/num_elems', '_07111_0/0/pressure_3/type', '_07111_0/0/pressure_3/value', '_07111_0/0/temperature_3/compressed_size', '_07111_0/0/temperature_3/num_elems', '_07111_0/0/temperature_3/type', '_07111_0/0/temperature_3/value', '_07111_0/1/pressure_3/compressed_size', '_07111_0/1/pressure_3/num_elems', '_07111_0/1/pressure_3/type', '_07111_0/1/pressure_3/value', '_07111_0/1/temperature_3/compressed_size', '_07111_0/1/temperature_3/num_elems', '_07111_0/1/temperature_3/type', '_07111_0/1/temperature_3/value', '_07111_0/status', '_07111_1/0/pressure_3/compressed_size', '_07111_1/0/pressure_3/num_elems', '_07111_1/0/pressure_3/type', '_07111_1/0/pressure_3/value', '_07111_1/0/temperature_3/compressed_size', '_07111_1/0/temperature_3/num_elems', '_07111_1/0/temperature_3/type', '_07111_1/0/temperature_3/value', '_07111_1/1/pressure_3/compressed_size', '_07111_1/1/pressure_3/num_elems', '_07111_1/1/pressure_3/type', '_0711

['pressure_3',
 '_07111_4',
 '_07111_3',
 'temperature_3',
 '_07111_2',
 '_07111_0',
 '_07111_1']

['pressure_3',
 '_07111_4',
 '_07111_3',
 'temperature_3',
 '_07111_2',
 '_07111_0',
 '_07111_1']

In [19]:
get_value(dbs[0],"_07111_4/1/temperature_3/num_elems")

'50'

'50'

In [20]:
get_value(dbs[0],"_07111_4/1/pressure_3/num_elems")

'50'

'50'

In [21]:
com_x = get_decompDataBLOSC(dbs[0], "_07111_4/1/pressure_3/value", 50)

In [22]:
com_x

(50000.0,
 50040.0,
 50080.0,
 50120.0,
 50160.0,
 50200.0,
 50240.0,
 50280.0,
 50320.0,
 50360.0,
 50400.0,
 50440.0,
 50480.0,
 50520.0,
 50560.0,
 50600.0,
 50640.0,
 50680.0,
 50720.0,
 50760.0,
 50800.0,
 50840.0,
 50880.0,
 50920.0,
 50960.0,
 51000.0,
 51040.0,
 51080.0,
 51120.0,
 51160.0,
 51200.0,
 51240.0,
 51280.0,
 51320.0,
 51360.0,
 51400.0,
 51440.0,
 51480.0,
 51520.0,
 51560.0,
 51600.0,
 51640.0,
 51680.0,
 51720.0,
 51760.0,
 51800.0,
 51840.0,
 51880.0,
 51920.0,
 51960.0)

(50000.0,
 50040.0,
 50080.0,
 50120.0,
 50160.0,
 50200.0,
 50240.0,
 50280.0,
 50320.0,
 50360.0,
 50400.0,
 50440.0,
 50480.0,
 50520.0,
 50560.0,
 50600.0,
 50640.0,
 50680.0,
 50720.0,
 50760.0,
 50800.0,
 50840.0,
 50880.0,
 50920.0,
 50960.0,
 51000.0,
 51040.0,
 51080.0,
 51120.0,
 51160.0,
 51200.0,
 51240.0,
 51280.0,
 51320.0,
 51360.0,
 51400.0,
 51440.0,
 51480.0,
 51520.0,
 51560.0,
 51600.0,
 51640.0,
 51680.0,
 51720.0,
 51760.0,
 51800.0,
 51840.0,
 51880.0,
 51920.0,
 51960.0)

In [ ]:
len(com_x)

In [23]:
data = get_data(dbs[0], "_07111_4/1/temperature_3/value")

In [24]:
type(data)

bytearray

bytearray

In [25]:
len(data)

184

184

In [26]:
np_array = np.frombuffer(data, dtype=np.uint8)

In [27]:
type(np_array)

numpy.ndarray

numpy.ndarray

In [28]:
lib_extention = {
            "darwin": "libSZ3c.dylib",
            "windows": "SZ3c.dll",
        }.get(sys.platform, "libSZ3c.so")

sz = SZ("/projects/insituperf/SZ3/install/lib64/{}".format(lib_extention))

In [29]:
data_dec = sz.decompress(np_array, (50,1,1), np.float32)

In [30]:
data_dec

array([[[ 9991.816]],

       [[10034.5  ]],

       [[10076.853]],

       [[10118.865]],

       [[10160.561]],

       [[10201.987]],

       [[10243.065]],

       [[10283.771]],

       [[10323.991]],

       [[10363.505]],

       [[10402.592]],

       [[10441.442]],

       [[10480.113]],

       [[10518.679]],

       [[10557.261]],

       [[10595.928]],

       [[10634.906]],

       [[10674.55 ]],

       [[10714.545]],

       [[10754.715]],

       [[10795.014]],

       [[10835.334]],

       [[10875.691]],

       [[10916.087]],

       [[10956.451]],

       [[10996.691]],

       [[11036.879]],

       [[11077.054]],

       [[11117.224]],

       [[11157.414]],

       [[11197.61 ]],

       [[11237.805]],

       [[11277.997]],

       [[11318.189]],

       [[11358.383]],

       [[11398.576]],

       [[11438.77 ]],

       [[11478.962]],

       [[11519.156]],

       [[11559.35 ]],

       [[11599.543]],

       [[11639.736]],

       [[11679.93 ]],

       [[11

array([[[ 9991.816]],

       [[10034.5  ]],

       [[10076.853]],

       [[10118.865]],

       [[10160.561]],

       [[10201.987]],

       [[10243.065]],

       [[10283.771]],

       [[10323.991]],

       [[10363.505]],

       [[10402.592]],

       [[10441.442]],

       [[10480.113]],

       [[10518.679]],

       [[10557.261]],

       [[10595.928]],

       [[10634.906]],

       [[10674.55 ]],

       [[10714.545]],

       [[10754.715]],

       [[10795.014]],

       [[10835.334]],

       [[10875.691]],

       [[10916.087]],

       [[10956.451]],

       [[10996.691]],

       [[11036.879]],

       [[11077.054]],

       [[11117.224]],

       [[11157.414]],

       [[11197.61 ]],

       [[11237.805]],

       [[11277.997]],

       [[11318.189]],

       [[11358.383]],

       [[11398.576]],

       [[11438.77 ]],

       [[11478.962]],

       [[11519.156]],

       [[11559.35 ]],

       [[11599.543]],

       [[11639.736]],

       [[11679.93 ]],

       [[11

In [44]:
test_array = ['_07112/0/0/pressure_3/compressed_size', '_07112/0/0/pressure_3/num_elems', '_07112/0/0/pressure_3/type', '_07112/0/0/pressure_3/value', '_07112/0/0/temperature_3/compressed_size', '_07112/0/0/temperature_3/num_elems', '_07112/0/0/temperature_3/type', '_07112/0/0/temperature_3/value', '_07112/0/1/pressure_3/compressed_size', '_07112/0/1/pressure_3/num_elems', '_07112/0/1/pressure_3/type', '_07112/0/1/pressure_3/value', '_07112/0/1/temperature_3/compressed_size', '_07112/0/1/temperature_3/num_elems', '_07112/0/1/temperature_3/type', '_07112/0/1/temperature_3/value', '_07112/1/0/pressure_3/compressed_size', '_07112/1/0/pressure_3/num_elems', '_07112/1/0/pressure_3/type', '_07112/1/0/pressure_3/value', '_07112/1/0/temperature_3/compressed_size', '_07112/1/0/temperature_3/num_elems', '_07112/1/0/temperature_3/type', '_07112/1/0/temperature_3/value', '_07112/1/1/pressure_3/compressed_size', '_07112/1/1/pressure_3/num_elems', '_07112/1/1/pressure_3/type', '_07112/1/1/pressure_3/value', '_07112/1/1/temperature_3/compressed_size', '_07112/1/1/temperature_3/num_elems', '_07112/1/1/temperature_3/type', '_07112/1/1/temperature_3/value', '_07112/2/0/pressure_3/compressed_size', '_07112/2/0/pressure_3/num_elems', '_07112/2/0/pressure_3/type', '_07112/2/0/pressure_3/value', '_07112/2/0/temperature_3/compressed_size', '_07112/2/0/temperature_3/num_elems', '_07112/2/0/temperature_3/type', '_07112/2/0/temperature_3/value', '_07112/2/1/pressure_3/compressed_size', '_07112/2/1/pressure_3/num_elems', '_07112/2/1/pressure_3/type', '_07112/2/1/pressure_3/value', '_07112/2/1/temperature_3/compressed_size', '_07112/2/1/temperature_3/num_elems', '_07112/2/1/temperature_3/type', '_07112/2/1/temperature_3/value', '_07112/3/0/pressure_3/compressed_size', '_07112/3/0/pressure_3/num_elems', '_07112/3/0/pressure_3/type', '_07112/3/0/pressure_3/value', '_07112/3/0/temperature_3/compressed_size', '_07112/3/0/temperature_3/num_elems', '_07112/3/0/temperature_3/type', '_07112/3/0/temperature_3/value', '_07112/3/1/pressure_3/compressed_size', '_07112/3/1/pressure_3/num_elems', '_07112/3/1/pressure_3/type', '_07112/3/1/pressure_3/value', '_07112/3/1/temperature_3/compressed_size', '_07112/3/1/temperature_3/num_elems', '_07112/3/1/temperature_3/type', '_07112/3/1/temperature_3/value', '_07112/4/0/pressure_3/compressed_size', '_07112/4/0/pressure_3/num_elems', '_07112/4/0/pressure_3/type', '_07112/4/0/pressure_3/value', '_07112/4/0/temperature_3/compressed_size', '_07112/4/0/temperature_3/num_elems', '_07112/4/0/temperature_3/type', '_07112/4/0/temperature_3/value', '_07112/4/1/pressure_3/compressed_size', '_07112/4/1/pressure_3/num_elems', '_07112/4/1/pressure_3/type', '_07112/4/1/pressure_3/value', '_07112/4/1/temperature_3/compressed_size', '_07112/4/1/temperature_3/num_elems', '_07112/4/1/temperature_3/type', '_07112/4/1/temperature_3/value', '_07112/0/status', '_07112/1/status', '_07112/2/tatus', '_07112/3/status', '_07112/4/status']

In [45]:
test_array

['_07112/0/0/pressure_3/compressed_size',
 '_07112/0/0/pressure_3/num_elems',
 '_07112/0/0/pressure_3/type',
 '_07112/0/0/pressure_3/value',
 '_07112/0/0/temperature_3/compressed_size',
 '_07112/0/0/temperature_3/num_elems',
 '_07112/0/0/temperature_3/type',
 '_07112/0/0/temperature_3/value',
 '_07112/0/1/pressure_3/compressed_size',
 '_07112/0/1/pressure_3/num_elems',
 '_07112/0/1/pressure_3/type',
 '_07112/0/1/pressure_3/value',
 '_07112/0/1/temperature_3/compressed_size',
 '_07112/0/1/temperature_3/num_elems',
 '_07112/0/1/temperature_3/type',
 '_07112/0/1/temperature_3/value',
 '_07112/1/0/pressure_3/compressed_size',
 '_07112/1/0/pressure_3/num_elems',
 '_07112/1/0/pressure_3/type',
 '_07112/1/0/pressure_3/value',
 '_07112/1/0/temperature_3/compressed_size',
 '_07112/1/0/temperature_3/num_elems',
 '_07112/1/0/temperature_3/type',
 '_07112/1/0/temperature_3/value',
 '_07112/1/1/pressure_3/compressed_size',
 '_07112/1/1/pressure_3/num_elems',
 '_07112/1/1/pressure_3/type',
 '_07112/

['_07112/0/0/pressure_3/compressed_size',
 '_07112/0/0/pressure_3/num_elems',
 '_07112/0/0/pressure_3/type',
 '_07112/0/0/pressure_3/value',
 '_07112/0/0/temperature_3/compressed_size',
 '_07112/0/0/temperature_3/num_elems',
 '_07112/0/0/temperature_3/type',
 '_07112/0/0/temperature_3/value',
 '_07112/0/1/pressure_3/compressed_size',
 '_07112/0/1/pressure_3/num_elems',
 '_07112/0/1/pressure_3/type',
 '_07112/0/1/pressure_3/value',
 '_07112/0/1/temperature_3/compressed_size',
 '_07112/0/1/temperature_3/num_elems',
 '_07112/0/1/temperature_3/type',
 '_07112/0/1/temperature_3/value',
 '_07112/1/0/pressure_3/compressed_size',
 '_07112/1/0/pressure_3/num_elems',
 '_07112/1/0/pressure_3/type',
 '_07112/1/0/pressure_3/value',
 '_07112/1/0/temperature_3/compressed_size',
 '_07112/1/0/temperature_3/num_elems',
 '_07112/1/0/temperature_3/type',
 '_07112/1/0/temperature_3/value',
 '_07112/1/1/pressure_3/compressed_size',
 '_07112/1/1/pressure_3/num_elems',
 '_07112/1/1/pressure_3/type',
 '_07112/

In [46]:
x = []
for k in test_array:
    parts = k.split('/')
    name = parts[len(parts)-2]
    x.append(name)
x

['pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3'

['pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'temperature_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'pressure_3',
 'temperature_3'